# 🏠 Airbnb Madrid — Price Predictor
## Detailed EDA & Preprocessing

In [61]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [62]:
df = pd.read_csv('../data/listings.v2.csv')

In [63]:
df.shape

(25000, 79)

In [64]:
df.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [65]:
columns_to_keep = [
    'neighbourhood_cleansed', 'room_type', 'accommodates', 
    'bathrooms', 'bedrooms', 'beds', 'price',
    'minimum_nights', 'number_of_reviews', 
    'review_scores_rating', 'availability_365'
]
df = df[columns_to_keep]
df.shape

(25000, 11)

In [66]:
df.isnull().sum()

neighbourhood_cleansed       0
room_type                    0
accommodates                 0
bathrooms                 6040
bedrooms                  2512
beds                      6035
price                     6047
minimum_nights               0
number_of_reviews            0
review_scores_rating      5147
availability_365             0
dtype: int64

In [67]:
df = df.dropna()
df.shape

(15779, 11)

In [68]:
df['price'] = df['price'].str.replace('[$,]', '', regex = True).astype(float)

In [69]:
df['price'].describe()

count    15779.000000
mean       150.834273
std        455.942936
min          8.000000
25%         73.000000
50%        112.000000
75%        162.000000
max      25654.000000
Name: price, dtype: float64

In [70]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)

In [71]:
IQR = Q3 - Q1
upper_limit = Q3 + 1.5 * IQR

In [72]:
df = df[df['price'] <= upper_limit]
df.shape

(14824, 11)

In [73]:
df = df[(df['room_type'] == 'Entire home/apt') | (df['room_type'] == 'Private room')]
df.shape

(14680, 11)

In [74]:
df = df[df['availability_365'] != 0]
df.shape

(14517, 11)

In [75]:
top_neighbourhoods = df['neighbourhood_cleansed'].value_counts()[df['neighbourhood_cleansed'].value_counts() >= 139].index

In [76]:
df['neighbourhood_cleansed'] = df['neighbourhood_cleansed'].where(df['neighbourhood_cleansed'].isin(top_neighbourhoods), 'Other')

In [77]:
X = df.drop(['price'], axis=1)
y = df['price']

In [78]:
X = pd.get_dummies(X, columns=['neighbourhood_cleansed', 'room_type']).astype(int)

In [79]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [80]:
print(X_train.shape, X_test.shape)

(11613, 36) (2904, 36)


In [81]:
X_train.to_csv('../data/X_train_v2.csv', index = False)
X_test.to_csv('../data/X_test_v2.csv', index = False)
y_train.to_csv('../data/y_train_v2.csv', index = False)
y_test.to_csv('../data/y_test_v2.csv', index = False)